In [ ]:
def main(datasources, start_date, end_date):
    import gc
    import warnings
    import numpy as np
    import pandas as pd
    import dai

    warnings.filterwarnings("ignore")

    try:
        import lightgbm as lgb
    except Exception as e:
        raise ImportError("当前环境缺少 lightgbm，无法运行本因子。") from e

    bar1m_table = datasources["bar1m"]

    IC_THRESHOLD = 0.01
    MIN_SELECTED_STOCK_FEATURES = 10

    TRAIN_HISTORY_START = "2019-01-01 00:00:00"
    FIXED_TRAIN_END = "2023-12-31 23:59:59"
    TRAIN_BAR1M_TABLE = "bigalpha_2026_stock_bar1m"
    TRAIN_MONTH_STEP = 2
    RECENT_FULL_MONTHS = 8
    LOOKBACK_DAYS = 45

    RANDOM_SEED = 2026
    DOWN_MARKET_WEIGHT = 1.5
    SEVERE_REGIME_WEIGHT = 2.5

    DEFENSIVE_MIX_WEIGHT = 0.45

    DEFENSIVE_DIRECTION = {
        "daily_volatility_20d": -1.0,
        "downside_volatility_20d": -1.0,
        "average_tail_amount_share_20d": 1.0,
        "average_close_position_20d": 1.0,
    }

    stock_candidate_features = [
        "stock_return_1d",
        "overnight_gap_return_1d",
        "intraday_return_1d",
        "intraday_amplitude_1d",
        "close_position_1d",
        "tail_30m_return_1d",
        "tail_amount_share_1d",
        "tail_volume_share_1d",
        "reverse_tail_price_amount_momentum_1d",
        "log_amount_proxy_1d",
        "log_volume_proxy_1d",

        "return_5d",
        "return_20d",
        "return_cut_5d",
        "return_cut_20d",
        "daily_volatility_5d",
        "daily_volatility_20d",
        "downside_volatility_5d",
        "downside_volatility_20d",
        "trend_efficiency_5d",
        "trend_efficiency_20d",
        "idiosyncratic_volatility_20d",
        "average_close_position_5d",
        "average_close_position_20d",
        "average_tail_amount_share_5d",
        "average_tail_amount_share_20d",
        "average_tail_30m_return_5d",
        "average_tail_30m_return_20d",
        "average_reverse_tail_price_amount_momentum_5d",
        "average_reverse_tail_price_amount_momentum_20d",
        "average_amount_proxy_5d",
        "average_amount_proxy_20d",
        "average_volume_proxy_5d",
        "average_volume_proxy_20d",
        "amount_proxy_late_vs_early_5d",
        "amount_proxy_late_vs_early_20d",
        "volume_proxy_late_vs_early_5d",
        "volume_proxy_late_vs_early_20d",
    ]

    base_market_features = [
        "market_return",
        "market_ret_3d",
        "market_ret_5d",
        "market_ret_20d",
        "market_vol_5d",
        "market_vol_20d",
        "market_stress_3d",
        "market_stress_20d",
        "market_intraday_amplitude",
        "market_close_position",
        "market_tail_amount_share",
        "market_up_ratio",
        "market_down_ratio",
        "market_down_ratio_3d",
        "market_down_ratio_5d",
    ]

    regime_features = [
        "severe_down_regime",
        "severe_down_score",
    ]

    market_features = base_market_features + regime_features

    sd_ts = pd.to_datetime(start_date)
    ed_ts = pd.to_datetime(end_date)
    train_end = pd.to_datetime(FIXED_TRAIN_END)
    train_history_start = pd.to_datetime(TRAIN_HISTORY_START)

    def fmt(x):
        return pd.to_datetime(x).strftime("%Y-%m-%d %H:%M:%S")

    def read_stock_pool(sd, ed):
        df = dai.query(
            "SELECT date, instrument FROM bigalpha_2026_instruments",
            filters={"date": [fmt(sd), fmt(ed)]},
            compression=True,
        ).df()
        df["date"] = pd.to_datetime(df["date"])
        df["instrument"] = df["instrument"].astype(str)
        return df

    def read_daily_bar_features(table, sd, ed):
        sql = f"""
        WITH raw AS (
            SELECT
                date,
                date::DATE::DATETIME AS trade_date,
                instrument,
                open,
                high,
                low,
                close,
                pre_close,
                amount,
                volume,
                EXTRACT(HOUR FROM date) AS hh,
                EXTRACT(MINUTE FROM date) AS mm
            FROM {table}
        ),
        daily_base AS (
            SELECT
                trade_date AS date,
                instrument,
                ARG_MIN(open, date) AS open_first,
                ARG_MAX(close, date) AS close_last,
                ARG_MAX(pre_close, date) AS pre_close,
                MAX(high) AS high_day,
                MIN(low) AS low_day,
                SUM(amount) AS amount_proxy,
                SUM(volume) AS volume_proxy
            FROM raw
            GROUP BY trade_date, instrument
        ),
        tail_agg AS (
            SELECT
                trade_date AS date,
                instrument,
                ARG_MIN(open, date) AS tail_open,
                ARG_MAX(close, date) AS tail_close,
                SUM(amount) AS tail_amount,
                SUM(volume) AS tail_volume
            FROM raw
            WHERE (hh = 14 AND mm >= 30) OR (hh = 15 AND mm = 0)
            GROUP BY trade_date, instrument
        ),
        tail_30m AS (
            SELECT
                date,
                instrument,
                CASE
                    WHEN tail_open IS NULL OR tail_open <= 0 THEN NULL
                    ELSE tail_close / tail_open - 1.0
                END AS tail_30m_ret,
                COALESCE(tail_amount, 0) AS tail_30m_amount,
                COALESCE(tail_volume, 0) AS tail_30m_volume,
                CASE
                    WHEN tail_open IS NULL OR tail_open <= 0 THEN NULL
                    ELSE
                        (tail_close / tail_open - 1.0)
                        * LN(
                            1.0 + COALESCE(tail_amount, 0)
                        )
                END AS tail_price_amount_momentum
            FROM tail_agg
        )
        SELECT
            d.*,
            t.tail_30m_ret,
            t.tail_30m_amount,
            t.tail_30m_volume,
            t.tail_price_amount_momentum
        FROM daily_base d
        LEFT JOIN tail_30m t
        USING (date, instrument)
        ORDER BY d.date, d.instrument
        """
        df = dai.query(sql, filters={"date": [fmt(sd), fmt(ed)]}, compression=True).df()
        df["date"] = pd.to_datetime(df["date"])
        df["instrument"] = df["instrument"].astype(str)

        for c in df.columns:
            if c not in ["date", "instrument"]:
                df[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
                df[c] = df[c].astype("float32")
        return df

    def month_ranges(start_ts, end_ts):
        cur = pd.to_datetime(start_ts)
        end_ts = pd.to_datetime(end_ts)
        while cur <= end_ts:
            month_end = cur.to_period("M").to_timestamp("M") + pd.Timedelta(hours=23, minutes=59, seconds=59)
            chunk_end = min(month_end, end_ts)
            yield cur, chunk_end
            cur = chunk_end.normalize() + pd.Timedelta(days=1)

    def select_training_months():
        months = []
        first_month = train_history_start.to_period("M").to_timestamp()
        last_month = train_end.to_period("M").to_timestamp()
        recent_start = last_month - pd.DateOffset(months=RECENT_FULL_MONTHS - 1)

        cur = first_month
        i = 0
        while cur <= last_month:
            ms = max(cur, train_history_start)
            me = min(
                cur.to_period("M").to_timestamp("M") + pd.Timedelta(hours=23, minutes=59, seconds=59),
                train_end,
            )
            use_month = (i % TRAIN_MONTH_STEP == 0) or (cur >= recent_start)
            if use_month and me >= ms:
                months.append((ms, me))
            cur = cur + pd.DateOffset(months=1)
            i += 1
        return months

    def add_features(df):
        df = df.sort_values(["instrument", "date"]).copy()

        df["stock_return"] = np.where(
            df["pre_close"] > 0,
            df["close_last"] / df["pre_close"] - 1.0,
            np.nan,
        ).astype("float32")
        df["stock_return_1d"] = df["stock_return"]

        df["overnight_gap_return_1d"] = np.where(
            df["pre_close"] > 0,
            df["open_first"] / df["pre_close"] - 1.0,
            np.nan,
        ).astype("float32")

        df["intraday_return_1d"] = np.where(
            df["open_first"] > 0,
            df["close_last"] / df["open_first"] - 1.0,
            np.nan,
        ).astype("float32")

        df["intraday_amplitude"] = np.where(
            df["pre_close"] > 0,
            (df["high_day"] - df["low_day"]) / df["pre_close"],
            np.nan,
        ).astype("float32")
        df["intraday_amplitude_1d"] = df["intraday_amplitude"]

        df["close_position"] = np.where(
            df["high_day"] > df["low_day"],
            (df["close_last"] - df["low_day"]) / (df["high_day"] - df["low_day"]),
            np.nan,
        ).astype("float32")
        df["close_position_1d"] = df["close_position"]

        df["tail_amount_share"] = np.where(
            df["amount_proxy"] > 0,
            df["tail_30m_amount"] / df["amount_proxy"],
            np.nan,
        ).astype("float32")
        df["tail_amount_share_1d"] = df["tail_amount_share"]

        df["tail_volume_share_1d"] = np.where(
            df["volume_proxy"] > 0,
            df["tail_30m_volume"] / df["volume_proxy"],
            np.nan,
        ).astype("float32")

        df["tail_30m_return_1d"] = df["tail_30m_ret"].astype("float32")
        df["reverse_tail_price_amount_momentum_1d"] = (-df["tail_price_amount_momentum"]).astype("float32")
        df["log_amount_proxy_1d"] = np.log1p(df["amount_proxy"].clip(lower=0)).astype("float32")
        df["log_volume_proxy_1d"] = np.log1p(df["volume_proxy"].clip(lower=0)).astype("float32")

        df["log_amount_proxy"] = df["log_amount_proxy_1d"]
        df["log_volume_proxy"] = df["log_volume_proxy_1d"]
        df["reverse_tail_price_amount_momentum"] = df["reverse_tail_price_amount_momentum_1d"]

        market = (
            df.groupby("date")
            .agg(
                market_return=("stock_return", "mean"),
                market_intraday_amplitude=("intraday_amplitude", "mean"),
                market_close_position=("close_position", "mean"),
                market_tail_amount_share=("tail_amount_share", "mean"),
                market_up_ratio=("stock_return", lambda x: np.mean(x > 0)),
                market_down_ratio=("stock_return", lambda x: np.mean(x < 0)),
            )
            .reset_index()
            .sort_values("date")
        )

        market["market_ret_3d"] = market["market_return"].rolling(3, min_periods=2).mean()
        market["market_ret_5d"] = market["market_return"].rolling(5, min_periods=3).mean()
        market["market_ret_20d"] = market["market_return"].rolling(20, min_periods=10).mean()
        market["market_vol_5d"] = market["market_return"].rolling(5, min_periods=3).std()
        market["market_vol_20d"] = market["market_return"].rolling(20, min_periods=10).std()
        market["market_down_ratio_3d"] = market["market_down_ratio"].rolling(3, min_periods=2).mean()
        market["market_down_ratio_5d"] = market["market_down_ratio"].rolling(5, min_periods=3).mean()
        market["market_stress_3d"] = -market["market_ret_3d"]
        market["market_stress_20d"] = -market["market_ret_20d"]

        for c in market.columns:
            if c != "date":
                market[c] = pd.to_numeric(market[c], errors="coerce").astype("float32")

        df = pd.merge(df, market, how="left", on="date")
        g = df.groupby("instrument", group_keys=False)

        for w in [5, 20]:
            minp = max(3, w // 2)

            df[f"return_{w}d"] = g["stock_return"].transform(
                lambda s: s.rolling(w, min_periods=minp).sum()
            )
            df[f"return_cut_{w}d"] = g["stock_return"].transform(
                lambda s: s.clip(-0.05, 0.05).rolling(w, min_periods=minp).sum()
            )
            df[f"daily_volatility_{w}d"] = g["stock_return"].transform(
                lambda s: s.rolling(w, min_periods=minp).std()
            )
            df[f"downside_volatility_{w}d"] = g["stock_return"].transform(
                lambda s: s.where(s < 0).rolling(w, min_periods=2).std()
            )
            df[f"trend_efficiency_{w}d"] = g["stock_return"].transform(
                lambda s: s.rolling(w, min_periods=minp).sum()
                / s.abs().rolling(w, min_periods=minp).sum().replace(0, np.nan)
            )
            df[f"average_close_position_{w}d"] = g["close_position"].transform(
                lambda s: s.rolling(w, min_periods=minp).mean()
            )
            df[f"average_tail_amount_share_{w}d"] = g["tail_amount_share"].transform(
                lambda s: s.rolling(w, min_periods=minp).mean()
            )
            df[f"average_tail_30m_return_{w}d"] = g["tail_30m_ret"].transform(
                lambda s: s.rolling(w, min_periods=minp).mean()
            )
            df[f"average_reverse_tail_price_amount_momentum_{w}d"] = g[
                "reverse_tail_price_amount_momentum"
            ].transform(lambda s: s.rolling(w, min_periods=minp).mean())
            df[f"average_amount_proxy_{w}d"] = g["log_amount_proxy"].transform(
                lambda s: s.rolling(w, min_periods=minp).mean()
            )
            df[f"average_volume_proxy_{w}d"] = g["log_volume_proxy"].transform(
                lambda s: s.rolling(w, min_periods=minp).mean()
            )

        for w in [5, 20]:
            half = max(2, w // 2)
            late_amount = g["log_amount_proxy"].transform(
                lambda s: s.rolling(half, min_periods=2).mean()
            )
            early_amount = g["log_amount_proxy"].transform(
                lambda s: s.rolling(half, min_periods=2).mean().shift(half)
            )
            late_volume = g["log_volume_proxy"].transform(
                lambda s: s.rolling(half, min_periods=2).mean()
            )
            early_volume = g["log_volume_proxy"].transform(
                lambda s: s.rolling(half, min_periods=2).mean().shift(half)
            )
            df[f"amount_proxy_late_vs_early_{w}d"] = late_amount - early_amount
            df[f"volume_proxy_late_vs_early_{w}d"] = late_volume - early_volume

        df["_ret_mkt"] = df["stock_return"] * df["market_return"]
        df["_mkt_sq"] = df["market_return"] * df["market_return"]
        g = df.groupby("instrument", group_keys=False)

        w = 20
        minp = 10
        mean_xy = g["_ret_mkt"].transform(lambda s: s.rolling(w, min_periods=minp).mean())
        mean_x = g["stock_return"].transform(lambda s: s.rolling(w, min_periods=minp).mean())
        mean_y = g["market_return"].transform(lambda s: s.rolling(w, min_periods=minp).mean())
        mean_yy = g["_mkt_sq"].transform(lambda s: s.rolling(w, min_periods=minp).mean())
        cov_xy = mean_xy - mean_x * mean_y
        var_y = mean_yy - mean_y * mean_y
        beta = cov_xy / var_y.replace(0, np.nan)
        resid = df["stock_return"] - beta * df["market_return"]

        df["idiosyncratic_volatility_20d"] = resid.groupby(df["instrument"]).transform(
            lambda s: s.rolling(w, min_periods=minp).std()
        )

        df = df.drop(columns=["_ret_mkt", "_mkt_sq"], errors="ignore")

        for c in df.columns:
            if c not in ["date", "instrument"]:
                df[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
                df[c] = df[c].astype("float32")

        return df

    def build_dataset_period(output_start, output_end, source_table=None):
        source_table = source_table or bar1m_table
        month_start = pd.to_datetime(output_start).to_period("M").to_timestamp()
        query_start = month_start - pd.Timedelta(days=LOOKBACK_DAYS)
        pool = read_stock_pool(output_start, output_end)
        raw = read_daily_bar_features(source_table, query_start, output_end)
        feat = add_features(raw)
        df = pd.merge(pool, feat, how="left", on=["date", "instrument"])
        df = df.sort_values(["date", "instrument"]).reset_index(drop=True)
        del pool, raw, feat
        gc.collect()
        return df

    def add_label(df):
        df = df.sort_values(["instrument", "date"]).copy()

        # Build the historical one-step-ahead label without a negative shift.
        # For each observed return date, map that return back to the instrument's
        # immediately preceding observed date. This preserves the intended
        # one-step-ahead training target while making the time direction explicit.
        label_source = df[["date", "instrument", "stock_return"]].copy()
        label_source["feature_date"] = label_source.groupby("instrument")["date"].shift(1)
        label_source = label_source.rename(
            columns={"date": "label_date", "feature_date": "date", "stock_return": "next_return"}
        )
        label_source = label_source.dropna(subset=["date"])

        df = pd.merge(
            df,
            label_source[["date", "instrument", "label_date", "next_return"]],
            how="left",
            on=["date", "instrument"],
            validate="one_to_one",
        )

        df["label"] = (
            df.groupby("date")["next_return"]
            .rank(pct=True, method="average")
            - 0.5
        )
        return df

    def fit_severe_thresholds(train_df):
        m = (
            train_df[
                [
                    "date",
                    "market_return",
                    "market_ret_3d",
                    "market_ret_5d",
                    "market_down_ratio",
                    "market_down_ratio_3d",
                    "market_down_ratio_5d",
                    "market_vol_5d",
                ]
            ]
            .drop_duplicates("date")
            .replace([np.inf, -np.inf], np.nan)
            .dropna(subset=["market_ret_3d", "market_ret_5d", "market_down_ratio_3d", "market_down_ratio_5d"])
        )

        return {
            "ret1_q10": float(m["market_return"].quantile(0.10)),
            "ret3_q20": float(m["market_ret_3d"].quantile(0.20)),
            "ret5_q20": float(m["market_ret_5d"].quantile(0.20)),
            "down1_q80": float(m["market_down_ratio"].quantile(0.80)),
            "down3_q70": float(m["market_down_ratio_3d"].quantile(0.70)),
            "down5_q70": float(m["market_down_ratio_5d"].quantile(0.70)),
            "vol5_q70": float(m["market_vol_5d"].quantile(0.70)),
        }

    def apply_severe_state(df, thresholds):
        df = df.copy()

        cond_5d = (
            (df["market_ret_5d"] <= thresholds["ret5_q20"])
            & (df["market_down_ratio_5d"] >= thresholds["down5_q70"])
        )
        cond_3d = (
            (df["market_ret_3d"] <= thresholds["ret3_q20"])
            & (df["market_down_ratio_3d"] >= thresholds["down3_q70"])
        )
        cond_1d = (
            (df["market_return"] <= thresholds["ret1_q10"])
            & (df["market_down_ratio"] >= thresholds["down1_q80"])
            & (df["market_vol_5d"] >= thresholds["vol5_q70"])
        )

        df["severe_down_regime"] = (cond_5d | cond_3d | cond_1d).astype("float32")

        score = (
            (-df["market_ret_3d"]).clip(lower=0).fillna(0.0) * 20.0
            + (-df["market_ret_5d"]).clip(lower=0).fillna(0.0) * 12.0
            + df["market_down_ratio_5d"].fillna(0.0)
            + df["market_vol_5d"].fillna(0.0) * 20.0
        )
        df["severe_down_score"] = score.astype("float32")

        return df

    def build_labeled_train_month(month_start, month_end):
        label_end = min(pd.to_datetime(month_end) + pd.Timedelta(days=10), train_end)
        df = build_dataset_period(month_start, label_end, TRAIN_BAR1M_TABLE)
        df = add_label(df)
        df = df[
            (df["date"] >= pd.to_datetime(month_start))
            & (df["date"] <= pd.to_datetime(month_end))
        ].copy()
        df = df.dropna(subset=["label"])

        invalid_label_time = (
            (df["label_date"] <= df["date"])
            | (df["label_date"] > train_end)
        )
        if invalid_label_time.any():
            raise ValueError("训练标签日期越界。")

        keep_cols = ["date", "instrument", "label"] + stock_candidate_features + base_market_features
        keep_cols = [c for c in keep_cols if c in df.columns]
        df = df[keep_cols].copy()

        gc.collect()
        return df

    def mean_rank_ic(df, feature):
        vals = []
        tmp = df[["date", feature, "label"]].dropna()
        for _, g0 in tmp.groupby("date"):
            if g0[feature].nunique() < 5 or g0["label"].nunique() < 5:
                continue
            ic = g0[feature].rank().corr(g0["label"].rank())
            if pd.notna(ic):
                vals.append(ic)
        if len(vals) == 0:
            return np.nan
        return float(np.mean(vals))

    def cs_z(s):
        sd = s.std()
        if pd.isna(sd) or sd == 0:
            return s * np.nan
        return (s - s.mean()) / sd

    def cs_rank(s):
        return s.rank(pct=True, method="average") - 0.5

    def defensive_score(df):
        parts = []

        specs = [
            ("daily_volatility_20d", -1.0),
            ("downside_volatility_20d", -1.0),
            ("average_tail_amount_share_20d", 1.0),
            ("average_close_position_20d", 1.0),
        ]

        for col, direction in specs:
            if col in df.columns:
                r = df.groupby("date")[col].transform(cs_rank)
                parts.append(direction * r)

        if len(parts) == 0:
            return pd.Series(0.0, index=df.index, dtype="float32")

        score = pd.concat(parts, axis=1).mean(axis=1)
        score = score.replace([np.inf, -np.inf], np.nan)
        score = score.groupby(df["date"]).transform(lambda s: s.fillna(s.median()))
        score = score.fillna(0.0).astype("float32")
        return score

    training_months = select_training_months()
    if len(training_months) == 0:
        raise ValueError("没有可用训练月份，请检查 start_date。")

    print("training months:", len(training_months))
    print("first train month:", fmt(training_months[0][0]), "~", fmt(training_months[0][1]))
    print("last train month:", fmt(training_months[-1][0]), "~", fmt(training_months[-1][1]))

    train_parts = []
    for i, (ms, me) in enumerate(training_months, 1):
        print(f"train month {i}/{len(training_months)}: {fmt(ms)} ~ {fmt(me)}")
        part = build_labeled_train_month(ms, me)
        if not part.empty:
            train_parts.append(part)
        del part
        gc.collect()

    if len(train_parts) == 0:
        raise ValueError("训练集为空。")

    train = pd.concat(train_parts, ignore_index=True)
    del train_parts
    gc.collect()

    train = train.replace([np.inf, -np.inf], np.nan)

    severe_thresholds = fit_severe_thresholds(train)
    train = apply_severe_state(train, severe_thresholds)

    print("severe thresholds:")
    print(severe_thresholds)
    print("train rows:", len(train))
    print("severe train rows:", int(train["severe_down_regime"].sum()))

    down_train = train[train["severe_down_regime"] > 0.5].copy()

    ic_rows = []
    for f in stock_candidate_features:
        if f in train.columns:
            full_ic = mean_rank_ic(train, f)
            severe_ic = mean_rank_ic(down_train, f) if not down_train.empty else np.nan
            ic_rows.append((f, full_ic, severe_ic))

    ic_table = pd.DataFrame(ic_rows, columns=["feature", "full_mean_rank_ic", "severe_mean_rank_ic"]).dropna(
        subset=["full_mean_rank_ic"],
        how="all",
    )
    ic_table["full_abs_ic"] = ic_table["full_mean_rank_ic"].abs()
    ic_table["severe_abs_ic"] = ic_table["severe_mean_rank_ic"].abs()
    ic_table["best_abs_ic"] = ic_table[["full_abs_ic", "severe_abs_ic"]].max(axis=1)
    ic_table = ic_table.sort_values("best_abs_ic", ascending=False).reset_index(drop=True)

    selected_stock_features = ic_table.loc[
        (ic_table["full_abs_ic"] >= IC_THRESHOLD)
        | (ic_table["severe_abs_ic"] >= IC_THRESHOLD),
        "feature",
    ].tolist()

    if len(selected_stock_features) < MIN_SELECTED_STOCK_FEATURES:
        for f in ic_table.head(MIN_SELECTED_STOCK_FEATURES)["feature"].tolist():
            if f not in selected_stock_features:
                selected_stock_features.append(f)

    for f in DEFENSIVE_DIRECTION:
        if f in train.columns and f not in selected_stock_features:
            selected_stock_features.append(f)

    selected_stock_features = [f for f in selected_stock_features if f in train.columns]
    selected_market_features = [f for f in market_features if f in train.columns]

    feature_direction = {}
    for _, r in ic_table.iterrows():
        full_ic = r["full_mean_rank_ic"]
        severe_ic = r["severe_mean_rank_ic"]
        if pd.notna(severe_ic) and abs(severe_ic) > abs(full_ic):
            use_ic = severe_ic
        else:
            use_ic = full_ic
        feature_direction[r["feature"]] = 1.0 if use_ic >= 0 else -1.0

    for f, d in DEFENSIVE_DIRECTION.items():
        feature_direction[f] = d

    def add_model_features(df, fit_stats=None):
        df = df.copy()
        stats = {} if fit_stats is None else dict(fit_stats)

        for f in selected_stock_features:
            if f not in df.columns:
                df[f] = np.nan

            direction = feature_direction.get(f, 1.0)
            z_col = f"z_{f}"
            r_col = f"r_{f}"

            df[z_col] = direction * df.groupby("date")[f].transform(cs_z)
            df[r_col] = direction * df.groupby("date")[f].transform(cs_rank)

        for f in selected_market_features:
            if f not in df.columns:
                df[f] = np.nan

        interaction_cols = []

        for f in selected_stock_features:
            z_col = f"z_{f}"
            r_col = f"r_{f}"

            if z_col in df.columns:
                c = f"{z_col}_x_severe"
                df[c] = df[z_col] * df["severe_down_regime"]
                interaction_cols.append(c)

            if r_col in df.columns:
                c = f"{r_col}_x_severe"
                df[c] = df[r_col] * df["severe_down_regime"]
                interaction_cols.append(c)

        for f in DEFENSIVE_DIRECTION:
            z_col = f"z_{f}"
            if z_col in df.columns:
                c1 = f"{z_col}_x_market_stress_3d"
                c2 = f"{z_col}_x_market_vol_20d"
                df[c1] = df[z_col] * df["market_stress_3d"]
                df[c2] = df[z_col] * df["market_vol_20d"]
                interaction_cols.extend([c1, c2])

        model_features = []
        for f in selected_stock_features:
            model_features.extend([f"z_{f}", f"r_{f}"])

        model_features += selected_market_features
        model_features += interaction_cols
        model_features = list(dict.fromkeys([f for f in model_features if f in df.columns]))

        if fit_stats is None:
            for f in model_features:
                med = pd.to_numeric(df[f], errors="coerce").replace([np.inf, -np.inf], np.nan).median()
                stats[f] = float(med) if pd.notna(med) else 0.0

        for f in model_features:
            fill_value = stats.get(f, 0.0)
            df[f] = pd.to_numeric(df[f], errors="coerce").replace([np.inf, -np.inf], np.nan)
            df[f] = df[f].fillna(fill_value).astype("float32")

        return df, model_features, stats

    train_m, model_features, fill_stats = add_model_features(train, fit_stats=None)

    X = train_m[model_features]
    y = train_m["label"]

    ok = y.notna()
    X = X.loc[ok]
    y = y.loc[ok]
    train_for_weight = train_m.loc[ok]

    if X.empty:
        raise ValueError("LightGBM 训练集为空。")

    market_ret = train_for_weight["market_return"].fillna(0.0)
    severe_flag = train_for_weight["severe_down_regime"].fillna(0.0)

    sample_weight = np.ones(len(train_for_weight), dtype="float32")
    sample_weight *= np.where(market_ret < 0, DOWN_MARKET_WEIGHT, 1.0).astype("float32")
    sample_weight *= np.where(severe_flag > 0.5, SEVERE_REGIME_WEIGHT, 1.0).astype("float32")
    sample_weight *= (1.0 + 12.0 * (-market_ret).clip(lower=0, upper=0.05)).astype("float32")

    model = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=180,
        learning_rate=0.035,
        num_leaves=15,
        max_depth=4,
        min_child_samples=220,
        subsample=0.72,
        subsample_freq=1,
        colsample_bytree=0.72,
        reg_alpha=0.8,
        reg_lambda=4.0,
        random_state=RANDOM_SEED,
        n_jobs=1,
        verbosity=-1,
    )

    model.fit(X, y, sample_weight=sample_weight)

    print("selected_stock_features:")
    print(selected_stock_features)
    print("top IC table:")
    print(ic_table.head(30))
    print("model_features:", len(model_features), "train_rows:", len(train_m))

    try:
        importance = pd.DataFrame({
            "feature": model_features,
            "importance": model.feature_importances_,
        }).sort_values("importance", ascending=False)
        print("feature importance top 30:")
        print(importance.head(30))
    except Exception:
        pass

    del train, train_m, X, y, train_for_weight
    gc.collect()

    outputs = []

    for cs, ce in month_ranges(sd_ts, ed_ts):
        print(f"predict chunk: {fmt(cs)} ~ {fmt(ce)}")
        pred = build_dataset_period(cs, ce)
        pred = apply_severe_state(pred, severe_thresholds)
        pred_m, _, _ = add_model_features(pred, fit_stats=fill_stats)

        pred_m["model_raw"] = model.predict(pred_m[model_features]).astype("float32")
        pred_m["model_score"] = pred_m.groupby("date")["model_raw"].rank(pct=True, method="average") - 0.5

        pred_m["defensive_score"] = defensive_score(pred_m)

        severe_gate = pred_m["severe_down_regime"].fillna(0.0).clip(0.0, 1.0) * DEFENSIVE_MIX_WEIGHT

        pred_m["raw_factor"] = (
            (1.0 - severe_gate) * pred_m["model_score"]
            + severe_gate * pred_m["defensive_score"]
        )

        pred_m["factor"] = pred_m.groupby("date")["raw_factor"].rank(pct=True, method="average") - 0.5
        pred_m["factor"] = pred_m.groupby("date")["factor"].transform(lambda s: s.fillna(s.median()))
        pred_m["factor"] = pred_m["factor"].fillna(0.0)

        outputs.append(pred_m[["date", "instrument", "factor"]].copy())

        print(
            "chunk severe days:",
            int(pred_m[["date", "severe_down_regime"]].drop_duplicates()["severe_down_regime"].sum()),
        )

        del pred, pred_m
        gc.collect()

    out = pd.concat(outputs, ignore_index=True)
    out = out.drop_duplicates(["date", "instrument"], keep="last")
    out = out.sort_values(["date", "instrument"]).reset_index(drop=True)

    out["factor"] = pd.to_numeric(out["factor"], errors="coerce").replace([np.inf, -np.inf], np.nan)
    out["factor"] = out["factor"].fillna(0.0).astype("float32")

    return out[["date", "instrument", "factor"]]
